In [ ]:
import os
os.chdir("..")

import pandas as pd
pd.set_option('display.precision', 2)
pd.set_option('display.float_format', '{:.2f}'.format)

from src.evaluation import extract_answers, compute_bleu, compute_rouge, compute_recall_at_k, compute_label_distribution, compute_human_vs_judge_agreement
from src.data_loader import load_json

## 1. Оценка качества генерации: Plain LLM vs RAG

In [2]:
results = load_json("data/llm_answers_labeled.json")

In [3]:
llm_predictions, references = extract_answers(results, "llm_answer")
rag_predictions, references = extract_answers(results, "rag_answer")

llm_bleu = compute_bleu(llm_predictions, references)
rag_bleu = compute_bleu(rag_predictions, references)

llm_rouge = compute_rouge(llm_predictions, references)
rag_rouge = compute_rouge(rag_predictions, references)

In [4]:
metrics = pd.DataFrame({"LLM": [llm_bleu, llm_rouge], "RAG": [rag_bleu, rag_rouge]}, index=['Bleu', 'Rouge-L'])

In [5]:
metrics

,LLM,RAG
Bleu,0.01,0.31
Rouge-L,0.02,0.80


## 2. Оценка качества Retriever

In [6]:
recall_1 = compute_recall_at_k(results, k=1)
recall_3 = compute_recall_at_k(results)

retrieval_quality = pd.DataFrame({"Recall@1": recall_1, "Recall@3": recall_3}, index=['metrics'])
retrieval_quality

,Recall@1,Recall@3
metrics,0.92,1.00


## 3. Оценка LLM as Judge vs Human Feedback

In [7]:
llm_as_judge_quality = compute_label_distribution(results)

llm_as_judge_df = pd.DataFrame({
            "Correct": llm_as_judge_quality['correct'],
            "Partial": llm_as_judge_quality['partial'],
            "Incorrect": llm_as_judge_quality['incorrect'],
            "Agreement": compute_human_vs_judge_agreement(results)['agreement']
        },
        index=['Label Distribution']
)
llm_as_judge_df

,Correct,Partial,Incorrect,Agreement
Label Distribution,0.94,0.04,0.02,0.98


## Заключение
* Plain LLM плохо отвечает без контекста.
* Использование RAG существенно улучшило качество генерации по сравнению с базовой моделью.
* В 92 % случаев ретривер возвращает первым самый релевантный документ
* В 100 % случаев самый релевантный документ попадает в топ-3
* Из 50 вопросов 47 были оценены как correct, 2 — как partial и 1 — как incorrect.
* Разметка человека и модели совпадает на 98 %